In [1]:
import os
os.chdir('/home/ec2-user/SageMaker')
os.system('git clone https://github.com/YinzhiZZZ/MLBA_credit-default-prediction')
os.system('aws s3 cp s3://credit-default-model-mlba/models/xgboost_model.pkl MLBA_credit-default-prediction/models/xgboost_model.pkl')
print("Setup complete!")

fatal: destination path 'MLBA_credit-default-prediction' already exists and is not an empty directory.


download: s3://credit-default-model-mlba/models/xgboost_model.pkl to MLBA_credit-default-prediction/models/xgboost_model.pkl
Setup complete!


In [3]:
import subprocess
result = subprocess.run(['conda', 'install', '-y', '-c', 'conda-forge', 'xgboost'], capture_output=True, text=True)
print("xgboost installed!")

xgboost installed!


In [4]:
import joblib, numpy as np, ipywidgets as widgets
from IPython.display import display, clear_output

model = joblib.load('MLBA_credit-default-prediction/models/xgboost_model.pkl')
out = widgets.Output()
btn = widgets.Button(description='Predict Default Risk', button_style='danger', layout=widgets.Layout(width='300px'))

style = {'description_width': 'initial'}
layout = widgets.Layout(width='500px')

s_limit = widgets.IntSlider(value=50000, min=10000, max=1000000, step=10000, description='Credit Limit (NT$):', style=style, layout=layout)
s_age   = widgets.IntSlider(value=30, min=18, max=75, description='Age:', style=style, layout=layout)
s_sex   = widgets.Dropdown(options=[('Female',2),('Male',1)], description='Gender:', style=style)
s_edu   = widgets.Dropdown(options=[('University',2),('Graduate School',1),('High School',3),('Other',4)], description='Education:', style=style)
s_mar   = widgets.Dropdown(options=[('Single',2),('Married',1),('Other',3)], description='Marital Status:', style=style)
s_pay0  = widgets.IntSlider(value=-1, min=-2, max=8, description='Pay Status Sep:', style=style, layout=layout)
s_pay2  = widgets.IntSlider(value=-1, min=-2, max=8, description='Pay Status Aug:', style=style, layout=layout)
s_pay3  = widgets.IntSlider(value=-1, min=-2, max=8, description='Pay Status Jul:', style=style, layout=layout)
s_bill1 = widgets.IntText(value=3913, description='Bill Amount Sep (NT$):', style=style)
s_bill2 = widgets.IntText(value=3102, description='Bill Amount Aug (NT$):', style=style)
s_amt1  = widgets.IntText(value=0,    description='Payment Sep (NT$):', style=style)
s_amt2  = widgets.IntText(value=689,  description='Payment Aug (NT$):', style=style)

def go(b):
    out.clear_output()
    with out:
        X = np.array([[
            s_limit.value, s_sex.value, s_edu.value, s_mar.value, s_age.value,
            s_pay0.value, s_pay2.value, s_pay3.value, -1, -1, -1,
            s_bill1.value, s_bill2.value, 0, 0, 0, 0,
            s_amt1.value, s_amt2.value, 0, 0, 0, 0
        ]])
        prob = float(model.predict_proba(X)[0][1])
        print("⚠️  HIGH RISK" if prob > 0.5 else "✅  LOW RISK")
        print(f"Default Probability: {prob:.1%}")

btn.on_click(go)
display(
    widgets.HTML("<h3>Customer Profile</h3>"),
    s_limit, s_age, s_sex, s_edu, s_mar,
    widgets.HTML("<h3>Payment Status (last 3 months)</h3><p>-2: No use | -1: Paid in full | 1~8: Months delayed</p>"),
    s_pay0, s_pay2, s_pay3,
    widgets.HTML("<h3>Bill & Payment Amounts</h3>"),
    s_bill1, s_bill2, s_amt1, s_amt2,
    btn, out
)

HTML(value='<h3>Customer Profile</h3>')

IntSlider(value=50000, description='Credit Limit (NT$):', layout=Layout(width='500px'), max=1000000, min=10000…

IntSlider(value=30, description='Age:', layout=Layout(width='500px'), max=75, min=18, style=SliderStyle(descri…

Dropdown(description='Gender:', options=(('Female', 2), ('Male', 1)), style=DescriptionStyle(description_width…

Dropdown(description='Education:', options=(('University', 2), ('Graduate School', 1), ('High School', 3), ('O…

...

IntText(value=3102, description='Bill Amount Aug (NT$):', style=DescriptionStyle(description_width='initial'))

IntText(value=0, description='Payment Sep (NT$):', style=DescriptionStyle(description_width='initial'))

IntText(value=689, description='Payment Aug (NT$):', style=DescriptionStyle(description_width='initial'))

Button(button_style='danger', description='Predict Default Risk', layout=Layout(width='300px'), style=ButtonSt…

Output()